In [1]:
!nvidia-smi

Tue May 19 09:27:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os

REPO_DIR = '/content/tb-classifier'
REPO_URL = 'https://github.com/vorrjjard-2/tb-classifier.git'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

Cloning into '/content/tb-classifier'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 173 (delta 67), reused 150 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 226.93 KiB | 25.21 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/tb-classifier


In [3]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATASET_NAME = 'tbx11k'  # canonical local name; must match data.root in the YAML
DRIVE_ZIP = f'/content/drive/MyDrive/datasets/zips/{DATASET_NAME}.zip'
DATA_DIR = Path(REPO_DIR) / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
target = DATA_DIR / DATASET_NAME

if not target.is_dir():
    # If a prior partial run left an extracted dir under a different name, pick it up
    # instead of re-unzipping. Otherwise copy + unzip from Drive.
    existing = [p for p in DATA_DIR.iterdir() if p.is_dir()]
    if not existing:
        !cp {DRIVE_ZIP} {DATA_DIR}/
        !unzip -q {DATA_DIR}/{DATASET_NAME}.zip -d {DATA_DIR}/
        !rm {DATA_DIR}/{DATASET_NAME}.zip
        existing = [p for p in DATA_DIR.iterdir() if p.is_dir()]
    if len(existing) != 1:
        raise RuntimeError(
            f"Expected exactly one dataset dir under {DATA_DIR}, got: "
            f"{[p.name for p in existing]}. Clean up manually."
        )
    if existing[0].name != DATASET_NAME:
        existing[0].rename(target)
        print(f"Renamed {existing[0].name}/ -> {DATASET_NAME}/")

!ls {target} | head

Mounted at /content/drive
Renamed tbx11k-classification-3-class-seed-42/ -> tbx11k/
test
train
val


In [4]:
!pip install -q -e . --no-deps
!pip install -q lightning torchmetrics albumentationsx opencv-python-headless \
    pydicom pyyaml wandb tqdm pillow pandas scikit-learn tensorboard

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tb-classifier (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 545.3/545.3 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ruben-saulog (models-ateneo-de-manila-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
!python scripts/train.py --config experiments/configs/default.yaml --fast-dev-run

Seed set to 42
  class_weights (balanced): [0.7368420958518982, 0.7368420958518982, 3.5]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist

## 7. Real training run

Checkpoints land in `/content/drive/MyDrive/tb-classifier-runs/resnet18-colab-001/checkpoints/` (configured in `resnet18_colab.yaml`), so they survive runtime disconnects.

In [ ]:
!python scripts/train.py --config experiments/configs/default.yaml

Seed set to 42
  class_weights (balanced): [0.7368420958518982, 0.7368420958518982, 3.5]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Current

## 8. Resume from checkpoint

If the runtime crashed mid-training, this cell picks the highest-epoch checkpoint from the run dir on Drive and continues until `epochs` in the YAML (currently 100). Optimizer and cosine LR scheduler state are restored, so the LR picks up exactly where it left off — no reset.

In [ ]:
import re
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/tb-classifier-runs/flipr-resnet1-npt-colab-001/checkpoints')
ckpts = list(ckpt_dir.glob('epoch=*.ckpt'))
assert ckpts, f'no checkpoints found in {ckpt_dir}'
latest = max(ckpts, key=lambda p: int(re.search(r'epoch=(\d+)', p.name).group(1)))
print('resuming from:', latest)

!python scripts/train.py --config experiments/configs/default.yaml --resume "{latest}"